# GazeNet v17 — Training Notebook (Colab)

GazeNet with geometric feature branch (iris ratios, head pose, z-depth).

**Prerequisites:**
- Google Drive contains: `210/gaze_wds_balanced/`, `210/gaze_labels.csv`, `210/geo_features_v1.parquet`
- Runtime set to **T4 GPU**

In [ ]:
# ============================================================
# INSTALL & MOUNT
# ============================================================
!pip install webdataset -q

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# ALL PATHS — everything references these variables
# ============================================================

TAR_DIR         = "/content/gaze_wds_balanced"
LABELS_CSV      = "/content/drive/MyDrive/210/gaze_labels.csv"
GEO_PARQUET     = "/content/drive/MyDrive/210/geo_features_v1.parquet"
CHECKPOINT_PATH = "/content/drive/MyDrive/210/best_gazenet_v17.pth"

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os, glob, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import webdataset as wds
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report

print("Imports loaded")

In [ ]:
# ============================================================
# COPY TARS TO LOCAL COLAB DISK (~10 min)
# Reading from Drive during training is slow, so we copy first.
# ============================================================

for split in ['train', 'val', 'test']:
    local_dir = f'/content/gaze_wds_balanced/{split}'
    os.makedirs(local_dir, exist_ok=True)
    source = f'/content/drive/MyDrive/210/gaze_wds_balanced/{split}/'
    tar_files = sorted(glob.glob(source + '*.tar'))
    for f in tar_files:
        fname = os.path.basename(f)
        dest = f"{local_dir}/{fname}"
        if not os.path.exists(dest):
            os.system(f"cp '{f}' '{dest}'")
    count = len([f for f in os.listdir(local_dir) if f.endswith('.tar')])
    print(f"{split}: {count} tars copied")

In [ ]:
# ============================================================
# VERIFY FILES
# ============================================================

train_tar_urls = sorted(glob.glob(f"{TAR_DIR}/train/*.tar"))
val_tar_urls   = sorted(glob.glob(f"{TAR_DIR}/val/*.tar"))
test_tar_urls  = sorted(glob.glob(f"{TAR_DIR}/test/*.tar"))

print(f"Train tars: {len(train_tar_urls)}")
print(f"Val tars:   {len(val_tar_urls)}")
print(f"Test tars:  {len(test_tar_urls)}")
print(f"Labels CSV: {os.path.exists(LABELS_CSV)}")
print(f"Geo parquet: {os.path.exists(GEO_PARQUET)}")

In [ ]:
# ============================================================
# LOAD LABELS → label_lookup dict
# Maps sample key (e.g. "00003_000000") to label string
# ============================================================

label_map = {'Straight': 0, 'Up': 1, 'Down': 2, 'Left': 3, 'Right': 4}

df_labels = pd.read_csv(LABELS_CSV, dtype={'subject_id': str})
print(f"Loaded {len(df_labels)} labeled frames")

label_lookup = {}
for _, row in df_labels.iterrows():
    key = f"{row['subject_id']}_{int(row['frame_idx']):06d}"
    label_lookup[key] = row['label']

print(f"Label lookup: {len(label_lookup)} entries")

In [ ]:
# ============================================================
# LOAD GEO FEATURES → geo_lookup dict
# Maps sample key to numpy array of 7 features.
# Precomputed locally with MediaPipe, uploaded to Drive.
# ============================================================

df_geo = pd.read_parquet(GEO_PARQUET)
print(f"Loaded geo features: {len(df_geo)} rows")

geo_cols = ['left_iris_h', 'right_iris_h', 'iris_h_agreement',
            'head_yaw', 'head_pitch', 'z_tilt', 'z_nose_rel']

geo_lookup = {}
for _, row in df_geo.iterrows():
    features = row[geo_cols].values.astype(np.float32)
    geo_lookup[row['key']] = features

print(f"Geo lookup: {len(geo_lookup)} entries")

# Neutral default for samples missing from geo_lookup
# (roughly dataset means)
GEO_DEFAULT = np.array([0.5, 0.5, 0.0, 0.0, 0.35, -0.1, -0.26], dtype=np.float32)
print(f"Geo default: {GEO_DEFAULT}")

In [ ]:
# ============================================================
# IMAGE TRANSFORMS
#
# Training: augmentation (color jitter, grayscale, blur)
# Val/Test: clean (just resize and normalize)
#
# Eye images: resize to 48x48 (from 60x36 raw)
# Face images: already 112x112, no resize needed
# Normalization: mean=0.5, std=0.5 for all channels
# ============================================================

# ---- Training transforms (with augmentation) ----
eye_transform_aug = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((48, 48)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

face_transform_aug = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# ---- Val/Test transforms (no augmentation) ----
eye_transform_clean = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

face_transform_clean = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print("Transforms defined")

In [ ]:
# ============================================================
# MAKE_DATASET: WebDataset pipeline with geo features
#
# Changes from v16A:
#   - Looks up geo_features from geo_lookup dict
#   - Returns geo_features tensor alongside images and label
#   - Falls back to GEO_DEFAULT if sample key not in geo_lookup
# ============================================================

def make_dataset(tar_urls, eye_transform, face_transform, shuffle=True):
    """Create a WebDataset pipeline that returns images + geo features."""

    def filter_and_transform(sample):
        key = sample["__key__"]

        # ---- Label lookup (same as v16A) ----
        label_str = label_lookup.get(key)
        if label_str is None:
            return None

        # ---- Image transforms (same as v16A) ----
        face = np.array(sample["face.jpg"])
        left_eye = np.array(sample["left.jpg"])
        right_eye = np.array(sample["right.jpg"])

        if eye_transform:
            left_eye = eye_transform(left_eye)
            right_eye = eye_transform(right_eye)
        if face_transform:
            face = face_transform(face)

        # ---- NEW: geo feature lookup ----
        geo_features = geo_lookup.get(key)
        if geo_features is None:
            geo_features = GEO_DEFAULT.copy()

        label_idx = label_map[label_str]

        return {
            'left_eye': left_eye,
            'right_eye': right_eye,
            'face': face,
            'geo_features': torch.tensor(geo_features, dtype=torch.float32),
            'label': torch.tensor(label_idx, dtype=torch.long),
        }

    dataset = (
        wds.WebDataset(tar_urls, shardshuffle=1000 if shuffle else False)
        .shuffle(50000 if shuffle else 0)
        .decode("pil")
        .map(filter_and_transform)
        .select(lambda x: x is not None)
    )

    return dataset

print("make_dataset defined")

In [ ]:
# ============================================================
# GazeNetV17 — Model Definition
#
# Same CNN streams as v16A GazeNet, plus a small MLP branch
# that processes our 7 geometric features.
#
# Architecture:
#   left_eye image  -> Eye CNN (shared weights) -> 4608 dims -+
#   right_eye image -> Eye CNN (shared weights) -> 4608 dims -+
#   face image      -> Face CNN                 -> 2304 dims -+
#   7 geo features  -> MLP (7->64->64)          ->   64 dims -+  <- NEW
#                                                             +-> FC -> 5 classes
#                                                    total: 11584
# ============================================================

class GazeNetV17(nn.Module):
    def __init__(self, num_classes=5, geo_feat_dim=7):
        super(GazeNetV17, self).__init__()

        # ---- Eye CNN (shared weights for left and right) ----
        # Input: (batch, 3, 48, 48) -> Output: (batch, 128, 6, 6)
        # UNCHANGED from v16A
        self.eye_cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # 48x48 -> 48x48
            nn.ReLU(),
            nn.MaxPool2d(2),                               # -> 24x24

            nn.Conv2d(32, 64, kernel_size=3, padding=1),   # -> 24x24
            nn.ReLU(),
            nn.MaxPool2d(2),                               # -> 12x12

            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # -> 12x12
            nn.ReLU(),
            nn.MaxPool2d(2),                               # -> 6x6
        )
        # Flattened: 128 * 6 * 6 = 4608 per eye

        # ---- Face CNN ----
        # Input: (batch, 3, 112, 112) -> Output: (batch, 256, 3, 3)
        # UNCHANGED from v16A
        self.face_cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=7, stride=2, padding=3),  # 112->56
            nn.ReLU(),
            nn.MaxPool2d(2),                                        # ->28

            nn.Conv2d(32, 64, kernel_size=5, padding=2),            # ->28
            nn.ReLU(),
            nn.MaxPool2d(2),                                        # ->14

            nn.Conv2d(64, 128, kernel_size=3, padding=1),           # ->14
            nn.ReLU(),
            nn.MaxPool2d(2),                                        # ->7

            nn.Conv2d(128, 256, kernel_size=3, padding=1),          # ->7
            nn.ReLU(),
            nn.MaxPool2d(2),                                        # ->3
        )
        # Flattened: 256 * 3 * 3 = 2304

        # ---- NEW: Geometric feature branch ----
        # Takes our 7 hand-crafted features and maps them to 64 dims
        # so they're meaningful alongside the ~11K CNN features
        self.geo_mlp = nn.Sequential(
            nn.Linear(geo_feat_dim, 64),   # 7 -> 64
            nn.ReLU(),
            nn.Dropout(0.3),               # lighter dropout than FC layers
            nn.Linear(64, 64),             # 64 -> 64
            nn.ReLU(),
        )

        # ---- FC classifier ----
        # Concatenated input: 4608 + 4608 + 2304 + 64 = 11584
        self.fc = nn.Sequential(
            nn.Linear(4608 * 2 + 2304 + 64, 512),  # 11584 -> 512
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(512, 256),                     # 512 -> 256
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, num_classes),              # 256 -> 5
        )

    def forward(self, left_eye, right_eye, face, geo_features):
        # ---- CNN streams (unchanged from v16A) ----
        left_feat  = self.eye_cnn(left_eye).view(left_eye.size(0), -1)    # -> 4608
        right_feat = self.eye_cnn(right_eye).view(right_eye.size(0), -1)  # -> 4608
        face_feat  = self.face_cnn(face).view(face.size(0), -1)           # -> 2304

        # ---- NEW: geometric feature branch ----
        geo_feat = self.geo_mlp(geo_features)  # -> 64

        # ---- Concatenate all four streams and classify ----
        combined = torch.cat([left_feat, right_feat, face_feat, geo_feat], dim=1)  # -> 11584
        return self.fc(combined)

print("GazeNetV17 defined")

In [ ]:
# ============================================================
# CREATE DATASETS AND DATALOADERS
# Train: augmented transforms, shuffled
# Val/Test: clean transforms, no shuffle
# ============================================================

train_dataset = make_dataset(train_tar_urls, eye_transform_aug,   face_transform_aug,   shuffle=True)
val_dataset   = make_dataset(val_tar_urls,   eye_transform_clean, face_transform_clean, shuffle=False)
test_dataset  = make_dataset(test_tar_urls,  eye_transform_clean, face_transform_clean, shuffle=False)

train_loader = wds.WebLoader(train_dataset, batch_size=32, num_workers=2, pin_memory=True)
val_loader   = wds.WebLoader(val_dataset,   batch_size=32, num_workers=2, pin_memory=True)
test_loader  = wds.WebLoader(test_dataset,  batch_size=32, num_workers=2, pin_memory=True)

print("Datasets and loaders ready")

In [ ]:
# ============================================================
# MODEL, LOSS, OPTIMIZER, SCHEDULER
# Same hyperparameters as v16A:
#   - lr: 1e-4
#   - weight_decay: 5e-4
#   - scheduler: ReduceLROnPlateau, patience=3
#   - early stopping: patience=6
# ============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = GazeNetV17(num_classes=5).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=5e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

print("Model, loss, optimizer, scheduler ready")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================================================
# SMOKE TEST: Run one batch through the full pipeline
# Confirms: tar loading -> image transforms -> geo lookup -> model forward pass
# ============================================================

test_tar = [train_tar_urls[0]]
print(f"Testing with: {test_tar[0]}")

test_ds = make_dataset(test_tar, eye_transform_clean, face_transform_clean, shuffle=False)
smoke_loader = wds.WebLoader(test_ds, batch_size=4, num_workers=0)

batch = next(iter(smoke_loader))

print(f"\nBatch contents:")
print(f"  left_eye:     {batch['left_eye'].shape}")
print(f"  right_eye:    {batch['right_eye'].shape}")
print(f"  face:         {batch['face'].shape}")
print(f"  geo_features: {batch['geo_features'].shape}")
print(f"  label:        {batch['label'].shape}")

model.eval()
with torch.no_grad():
    outputs = model(
        batch['left_eye'].to(device),
        batch['right_eye'].to(device),
        batch['face'].to(device),
        batch['geo_features'].to(device),
    )

print(f"\nModel output: {outputs.shape}")
print(f"Pipeline works end-to-end!")

In [ ]:
# ============================================================
# TRAINING LOOP
#
# Same structure as v16A with two changes:
#   1. geo_features extracted from batch and passed to model
#   2. Model checkpoint saved as best_gazenet_v17.pth
# ============================================================

num_epochs = 20
best_val_loss = float('inf')

# Early stopping
patience = 6
patience_counter = 0

# Track metrics for loss curves
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"{'='*50}")

    # ============================================
    # TRAINING PHASE
    # ============================================
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    epoch_start = time.time()

    for batch_idx, batch in enumerate(train_loader):
        # Move data to device
        left_eye     = batch['left_eye'].to(device)
        right_eye    = batch['right_eye'].to(device)
        face         = batch['face'].to(device)
        geo_features = batch['geo_features'].to(device)
        labels       = batch['label'].to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(left_eye, right_eye, face, geo_features)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Track metrics
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

        # Print progress every 1000 batches
        if (batch_idx + 1) % 1000 == 0:
            elapsed = time.time() - epoch_start
            batches_done = batch_idx + 1
            eta_minutes = (elapsed / batches_done) * (7000 - batches_done) / 60
            print(f"  Batch {batches_done}/~7000 - Loss: {loss.item():.4f} - ETA: {eta_minutes:.1f} min")

    # Calculate training metrics
    avg_train_loss = train_loss / (batch_idx + 1)
    train_acc = 100 * train_correct / train_total
    epoch_time = time.time() - epoch_start

    # ============================================
    # VALIDATION PHASE
    # ============================================
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    val_batch_count = 0

    with torch.no_grad():
        for batch in val_loader:
            val_batch_count += 1
            left_eye     = batch['left_eye'].to(device)
            right_eye    = batch['right_eye'].to(device)
            face         = batch['face'].to(device)
            geo_features = batch['geo_features'].to(device)
            labels       = batch['label'].to(device)

            outputs = model(left_eye, right_eye, face, geo_features)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    # Calculate validation metrics
    avg_val_loss = val_loss / val_batch_count
    val_acc = 100 * val_correct / val_total

    # Save metrics
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    # Step scheduler
    scheduler.step(avg_val_loss)

    # Print epoch summary
    print(f"\n  Time: {epoch_time/60:.1f} min")
    print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"  Gap: {train_acc - val_acc:.2f}%")

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  Saved best model (val_loss={avg_val_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")

    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

print(f"\nTraining complete!")
print(f"Best validation loss: {best_val_loss:.4f}")

In [ ]:
# ============================================================
# LOSS CURVES
# Same plots as v16A: loss, accuracy, and train-val gap
# ============================================================

plt.figure(figsize=(15, 5))

# ---- Loss ----
plt.subplot(1, 3, 1)
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-o', label='Train Loss')
plt.plot(range(1, len(val_losses) + 1), val_losses, 'r-o', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

# ---- Accuracy ----
plt.subplot(1, 3, 2)
plt.plot(range(1, len(train_accuracies) + 1), train_accuracies, 'b-o', label='Train Acc')
plt.plot(range(1, len(val_accuracies) + 1), val_accuracies, 'r-o', label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

# ---- Overfitting gap ----
plt.subplot(1, 3, 3)
gap = [train_accuracies[i] - val_accuracies[i] for i in range(len(train_accuracies))]
plt.plot(range(1, len(gap) + 1), gap, 'g-o')
plt.xlabel('Epoch')
plt.ylabel('Accuracy Gap (%)')
plt.title('Train-Val Accuracy Gap (Overfitting Check)')
plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
plt.grid(True)

plt.tight_layout()
plt.show()

# ---- Summary ----
print(f"\nTotal epochs: {len(train_losses)}")
print(f"Best val loss: {min(val_losses):.4f} (Epoch {val_losses.index(min(val_losses)) + 1})")
print(f"Best val acc:  {max(val_accuracies):.2f}% (Epoch {val_accuracies.index(max(val_accuracies)) + 1})")
print(f"Final gap:     {train_accuracies[-1] - val_accuracies[-1]:.2f}%")

In [ ]:
# ============================================================
# TEST EVALUATION & CONFUSION MATRIX
# Load best checkpoint, run on test set, print classification
# report and confusion matrix — same as v16A.
# ============================================================

# Load best model
model.load_state_dict(torch.load(CHECKPOINT_PATH))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        left_eye     = batch['left_eye'].to(device)
        right_eye    = batch['right_eye'].to(device)
        face         = batch['face'].to(device)
        geo_features = batch['geo_features'].to(device)
        labels       = batch['label'].to(device)

        outputs = model(left_eye, right_eye, face, geo_features)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---- Confusion Matrix ----
label_names = ['Straight', 'Up', 'Down', 'Left', 'Right']
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('GazeNet v17 Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

# ---- Classification Report ----
print("\n" + "="*50)
print("Test Set Performance")
print("="*50)
print(classification_report(all_labels, all_preds, target_names=label_names))